# AgentShield: train the injection classifier (Kaggle, GPU)

Fine-tunes a multilingual transformer to tell **prompt-injection attacks** from **benign text** in English, Bangla and Banglish. The trained model replaces the heuristic detector in `agentshield/checks/classifier.py` (same `predict(text)` interface).

**On Kaggle:**
1. *Add data*: upload the repo's `datasets/` folder as a Kaggle dataset named `agentshield-dataset` (the four labeled `*.jsonl` files: `benign.jsonl`, `benign_own.jsonl`, `attacks.jsonl`, `external.jsonl`; `external_raw/` is not needed and is skipped if present).
2. *Settings*: Accelerator **GPU T4 x1**, Internet **on** (to download the base model).
3. *Run all*. Training takes a few minutes.
4. Download `/kaggle/working/agentshield-classifier.zip` (right panel, *Output*).

**Honesty rules:**
- The data is small (a few hundred items). Scores on the **test split only** count; training and validation scores are not results.
- `split_ids.json` records which items are in the test split. The local benchmark must evaluate the model on those items only, never on items it was trained on.

In [ ]:
# Settings
MODEL_NAME = "xlm-roberta-base"   # multilingual incl. Bangla; alternatives: "csebuetnlp/banglabert" (Bangla only)
DATA_DIR = "/kaggle/input/agentshield-dataset"
OUT_DIR = "/kaggle/working/agentshield-classifier"
SEED = 42
MAX_LEN = 256
EPOCHS = 6
LR = 2e-5
BATCH = 16

In [ ]:
import glob, json, os, random
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Load the labeled data

In [ ]:
rows = []
# Only the labeled files (benign, benign_own, attacks, external); external_raw/ is skipped on purpose.
for path in sorted(glob.glob(f"{DATA_DIR}/**/*.jsonl", recursive=True)):
    if "external_raw" in path.replace("\\", "/"):
        continue
    with open(path, encoding="utf-8") as f:
        rows += [json.loads(line) for line in f if line.strip()]
df = pd.DataFrame(rows)
df = df[df["label"].isin(["attack", "benign"])].drop_duplicates("id").reset_index(drop=True)
df["y"] = (df["label"] == "attack").astype(int)
print(len(df), "items")
print(df.groupby(["language", "label"]).size().unstack(fill_value=0))

## 2. Split 70 / 15 / 15, balanced by language and label

In [ ]:
strata = df["language"] + "_" + df["label"]
train_df, rest = train_test_split(df, test_size=0.30, stratify=strata, random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest["language"] + "_" + rest["label"], random_state=SEED)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part), dict(part.groupby(["language", "label"]).size()))
os.makedirs(OUT_DIR, exist_ok=True)
with open(f"{OUT_DIR}/split_ids.json", "w") as f:
    json.dump({"train": list(train_df["id"]), "val": list(val_df["id"]), "test": list(test_df["id"])}, f, indent=1)

## 3. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(part):
    ds = Dataset.from_pandas(part[["text", "y"]].rename(columns={"y": "labels"}), preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)

train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)

## 4. Train (class weights, because there are fewer attacks than benign items)

In [ ]:
counts = np.bincount(train_df["y"], minlength=2)
class_weights = torch.tensor(len(train_df) / (2 * counts), dtype=torch.float)
print("class weights (benign, attack):", class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(pred):
    y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
    tp = int(((y_pred == 1) & (y_true == 1)).sum()); fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "benign", 1: "attack"}, label2id={"benign": 0, "attack": 1})

common = dict(output_dir="/kaggle/working/checkpoints", learning_rate=LR, num_train_epochs=EPOCHS,
              per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, weight_decay=0.01,
              save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
              save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
try:
    args = TrainingArguments(eval_strategy="epoch", **common)
except TypeError:  # older transformers
    args = TrainingArguments(evaluation_strategy="epoch", **common)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 5. Test split only: per-language detection rate and false positive rate

In [ ]:
probs = torch.softmax(torch.tensor(trainer.predict(test_ds).predictions), dim=-1)[:, 1].numpy()
test_df = test_df.assign(p_attack=probs, flagged=probs >= 0.5)

report = {}
for language, part in [("all", test_df)] + list(test_df.groupby("language")):
    att, ben = part[part["y"] == 1], part[part["y"] == 0]
    report[language] = {"attacks_flagged": f"{int(att['flagged'].sum())}/{len(att)}",
                        "benign_flagged": f"{int(ben['flagged'].sum())}/{len(ben)}"}
print(json.dumps(report, indent=1))
with open(f"{OUT_DIR}/test_metrics.json", "w") as f:
    json.dump({"model": MODEL_NAME, "seed": SEED, "threshold": 0.5, "per_language": report}, f, indent=1)

## 6. Save the model for CPU inference and zip it

In [ ]:
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
import shutil
shutil.make_archive("/kaggle/working/agentshield-classifier", "zip", OUT_DIR)
print("Download /kaggle/working/agentshield-classifier.zip")

## After downloading
Unzip into the repo as `models/agentshield-classifier/` (the `models/` folder is git-ignored: too large for git). Then a model-backed detector with the same `predict(text)` interface replaces the heuristic, and the benchmark is re-run **on the test-split ids only**.